# The dataset is now public now. Enjoy!⭐⭐⭐

For each task this notebook takes the cheapest circuit we have verified with the unmodified official verifiers,
re-verifies every file here, and writes `/kaggle/working/submission.zip`.

| Task | Circuit | Ratio (default salt) |
|---|---|---:|
| invsqrt | busyaprime's file (mrmeows' 63-qubit build with his main lookup) | 20.229 |
| DQI | SomeJoint's 897-qubit decoder, modified by us | 58.921 |
| dfthcsos | ours, on Michael Rustad's and Ahmed Berat Özer's C46 factorization | 9.606 |

## Local check before publishing

Kaggle's own metric code (the `google-quantum-ai-metric` notebook, unchanged) run end to end on these exact files with two local salts gave **22.5383** and **22.5383** (split 0 below). The leaderboard measures again with its own salt.

| Task | Cost | Ratio (reference / cost) |
|---|---:|---:|
| invsqrt | 102664 | 20.2288 |
| dqi | 4.14781e+11 | 58.9213 |
| dfthcsos | 3.4934e+08 | 9.6055 |

## Re-verify every component here

Each file is copied from the release dataset and matched to its SHA-256; then the three official verifiers from the competition data run on it, unmodified, at their default shot counts.

In [1]:
import glob, hashlib, json, lzma, math, os, shutil, sys, time, zipfile, pathlib

def find_comp():
    for p in [os.environ.get('QCC_COMP_DIR', ''), '/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation',
              '/kaggle/input/google-verifiable-quantum-circuit-compilation']:
        if p and os.path.isdir(os.path.join(p, 'verifiers')):
            return p
    raise SystemExit('competition data not found')

def find_release():
    if os.environ.get('QCC_DS_DIR'):
        return os.environ['QCC_DS_DIR']
    hits = glob.glob('/kaggle/input/**/QCC_RELEASE_MANIFEST.json', recursive=True)
    if not hits:
        raise SystemExit('release dataset not found')
    return os.path.dirname(hits[0])

COMP, DS = find_comp(), find_release()
WORK = pathlib.Path(os.environ.get('QCC_WORK_DIR', '/kaggle/working'))
SUB = WORK / 'submission'
sys.path[:0] = [os.path.join(COMP, 'kickmix'), os.path.join(COMP, 'verifiers')]
import kickmix as km
from invsqrt import verify_submission_invsqrt as vinv
from dqi import verify_submission_dqi as vdqi
from dfthcsos import verify as vdft
MANIFEST = json.load(open(os.path.join(DS, 'QCC_RELEASE_MANIFEST.json')))
EXPECTED_PORTFOLIO = 'v13_p897'
assert MANIFEST['portfolio'] == EXPECTED_PORTFOLIO, ('stale dataset version', MANIFEST['portfolio'], EXPECTED_PORTFOLIO)
SALT = MANIFEST['salt']
print('competition data:', COMP, '| release dataset:', DS, '| salt:', SALT)

competition data: /kaggle/input/competitions/google-verifiable-quantum-circuit-compilation | release dataset: /kaggle/input/qcc-circuits-live | salt: notebook-verification


In [2]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

if SUB.exists():
    shutil.rmtree(SUB)
for item in MANIFEST['files']:
    src, dst = os.path.join(DS, item['stored_as']), SUB / item['file']
    dst.parent.mkdir(parents=True, exist_ok=True)
    if item['stored_as'].endswith('.xz'):
        with lzma.open(src) as fi, open(dst, 'wb') as fo:
            shutil.copyfileobj(fi, fo, 1 << 22)
    else:
        shutil.copyfile(src, dst)
    got = sha256(dst)
    assert got == item['sha256'], (item['file'], got)
print(len(MANIFEST['files']), 'files copied and matched to the manifest by SHA-256')

18 files copied and matched to the manifest by SHA-256


In [3]:
REF = MANIFEST['reference_costs']
results = {}
t0 = time.time()
circ = km.Circuit.from_file(str(SUB / 'invsqrt.kmb'), format='kmb')
results['invsqrt'] = vinv.verify(circ, vinv.ProblemSpecInvSqrt.for_n_p(10), salt=SALT)
print('invsqrt done', round(time.time() - t0), 's'); t0 = time.time()
circ = km.Circuit.from_file(str(SUB / 'dqi.kmb'), format='kmb')
results['dqi'] = vdqi.verify(circ, vdqi.ProblemSpecDQI.benchmark_instance(), salt=SALT)
del circ
print('dqi done', round(time.time() - t0), 's'); t0 = time.time()
results['dfthcsos'] = vdft.verify(str(SUB / 'dfthcsos'), salt=SALT)
print('dfthcsos done', round(time.time() - t0), 's')
rows, ratios = [], []
for task in ('invsqrt', 'dqi', 'dfthcsos'):
    r = results[task]
    assert r['status'] == 'PASS', (task, r)
    ratio = REF[task] / float(r['score'])
    ratios.append(ratio)
    if task == 'invsqrt':
        extra = f"{r['num_qubits']} qubits, {r['toffoli_equivalent']:.1f} Toffoli-equivalents"
    elif task == 'dqi':
        extra = (f"{r['num_qubits']} qubits, {r['toffoli_count']:.0f} mean Toffolis, "
                 f"{r.get('value_errors', 0)} value / {r.get('phase_errors', 0)} phase errors")
    else:
        extra = f"lambda_eff {r['lambda_eff']:.3f}, {r['toffoli_count']:.0f} Toffolis, {r['num_qubits']} qubits"
    rows.append((task, float(r['score']), ratio, extra))
    print(f"{task:9s} PASS cost {float(r['score']):.6g}  ratio {ratio:8.4f}  ({extra})")
gm = math.prod(ratios) ** (1 / 3)
print(f'geometric mean of reference / cost: {gm:.4f}')

invsqrt done 669 s
dqi done 123 s
dfthcsos done 26 s
invsqrt   PASS cost 102664  ratio  20.2288  (63 qubits, 1629.6 Toffoli-equivalents)
dqi       PASS cost 4.14781e+11  ratio  58.9214  (889 qubits, 440564 mean Toffolis, 0 value / 0 phase errors)
dfthcsos  PASS cost 3.4934e+08  ratio   9.6055  (lambda_eff 47.494, 17066 Toffolis, 431 qubits)
geometric mean of reference / cost: 22.5383


## Write the submission

In [4]:
zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in MANIFEST['files']:
        z.write(SUB / item['file'], item['file'])
shutil.rmtree(SUB)
print('wrote', zip_path, round(zip_path.stat().st_size / 2**20, 1), 'MiB')

wrote /kaggle/working/submission.zip 26.0 MiB


# Sources and credits

- **invsqrt** — Akmal Xodarev's (busyaprime) file from https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members
  (version 4, Apache-2.0; notebook https://www.kaggle.com/code/busyaprime/lb-22-33981-22x-cheaper-than-google-s-circuits),
  byte for byte: mrmeows' (team selenium) 63-qubit build (https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits,
  Apache-2.0) with busyaprime's re-synthesised main lookup. Lineage as they name it: Akmal Xodarev (busyaprime), Phuc
  Thai (DePie), Dmitrii Gluzdov, marsyas.
- **dqi** — generated from SomeJoint's (Agniv Banerjee's team) Apache-2.0 source for the 897-qubit decoder with
  GPT-6 Astra Pro's 658-Toffoli jet (https://www.kaggle.com/code/somejoint/quantum-circuits-22-34-local-gm-897-qubit-dqi),
  **modified by heliosli**. Lineage as its authors name it: GPT-6 Astra Pro and SomeJoint's research team; Ahmed Berat
  Özer; Alex Chilton's 883-qubit Schur decoder, based on jday96314's QuantumAI decoder
  (https://github.com/jday96314/QuantumAI); mrmeows, Ahmed Berat Özer and manwithacat (mask projection); Dmitrii
  Gluzdov, Agniv Banerjee (SomeJoint), ViberTeam, Phuc Thai (DePie).
- **dfthcsos** — built on Michael Rustad's R = 10, B = 129, C = 46 factorization and Ahmed Berat Özer's quantization
  (https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, version 9), with the lineage that
  notebook names: mrmeows, kozykappa, marculera, Vladislav Lavrentev (lightningv08). The ANGLE circuits and the final
  table weights are ours.
- **Organizers** — task definitions, verifiers and kickmix (Google Quantum AI / Qualtran, competition data).